# Analyze Results

Jupyter notebook for visualization and analysis of benchmark results.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Load Benchmark Results

Load the most recent benchmark results CSV file from the results directory.

In [2]:
# Find the most recent results file
results_dir = Path("../results")
csv_files = list(results_dir.glob("benchmark_results_*.csv"))

if not csv_files:
    print("❌ No results files found in ../results/")
    print("   Run the benchmark first: uv run python scripts/run_tier_sweep.py")
    df = None
else:
    # Get most recent file
    latest_file = max(csv_files, key=lambda p: p.stat().st_mtime)

    # Load data
    df = pd.read_csv(latest_file)

    # Convert boolean columns if needed
    if "USE_MEDIAN_FOR_OUTLIER_DETECTION" in df.columns:
        df["USE_MEDIAN_FOR_OUTLIER_DETECTION"] = df[
            "USE_MEDIAN_FOR_OUTLIER_DETECTION"
        ].fillna(False)

    print(f"✅ Loaded {len(df)} results from {latest_file.name}")
    print(f"\n**Tiers**: {', '.join(sorted(df['Tier'].unique().tolist()))}")
    print(f"**Methods**: {', '.join(sorted(df['Method'].unique().tolist()))}")
    print(f"**Status**: {dict(df['Status'].value_counts())}")

✅ Loaded 2 results from benchmark_results_20260211_170250.csv

**Tiers**: tier1
**Methods**: ALFESE, GEMSS
**Status**: {'Success': np.int64(2)}


## Data Overview

Examine the structure and summary statistics of the benchmark results.

In [3]:
if df is not None:
    # Filter successful results
    df_success = df[df["Status"] == "Success"].copy()

    print(f"📈 Successful Results: {len(df_success)}/{len(df)}")
    print(f"\n### Dataset Dimensions:")
    dimensions = df_success.groupby(["Tier", "Method"])[
        ["N_SAMPLES", "N_FEATURES"]
    ].first()
    display(dimensions)
    
    print(f"\n### Performance Metrics Summary:")
    metrics = ["Recall", "Precision", "Success_Index", "Adjusted_SI", "Runtime_Sec"]
    stats = df_success[metrics].describe().round(3)
    display(stats)
else:
    df_success = None
    print("⚠️ No data loaded")

📈 Successful Results: 2/2

### Dataset Dimensions:


N_SAMPLES  N_FEATURES
Tier  Method                       
tier1 ALFESE         50         200
      GEMSS          50         200


### Performance Metrics Summary:


,Recall,Precision,Success_Index,Adjusted_SI,Runtime_Sec
count,2.000,2.000,2.000,2.000,2.000
mean,0.964,0.559,13.776,7.920,255.837
std,0.051,0.624,0.722,9.003,340.691
min,0.929,0.117,13.265,1.554,14.932
25%,0.946,0.338,13.520,4.737,135.384
50%,0.964,0.559,13.776,7.920,255.837
75%,0.982,0.779,14.031,11.103,376.289
max,1.000,1.000,14.286,14.286,496.742


## Performance Comparison

Compare methods across different metrics (Recall, Precision, Adjusted SI).

In [4]:
if df is not None and len(df_success) > 0:
    # Create subplots
    fig_perf = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Recall by Method and Tier",
            "Precision by Method and Tier",
            "Adjusted Success Index by Method and Tier",
            "Runtime Distribution by Method",
        ),
        specs=[
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "box"}],
        ],
    )

    # Recall by Method and Tier
    df_pivot_recall = df_success.pivot_table(
        values="Recall", index="Method", columns="Tier", aggfunc="mean"
    )
    for tier in df_pivot_recall.columns:
        fig_perf.add_trace(
            go.Bar(name=tier, x=df_pivot_recall.index, y=df_pivot_recall[tier]),
            row=1,
            col=1,
        )

    # Precision by Method and Tier
    df_pivot_prec = df_success.pivot_table(
        values="Precision", index="Method", columns="Tier", aggfunc="mean"
    )
    for tier in df_pivot_prec.columns:
        fig_perf.add_trace(
            go.Bar(
                name=tier,
                x=df_pivot_prec.index,
                y=df_pivot_prec[tier],
                showlegend=False,
            ),
            row=1,
            col=2,
        )

    # Adjusted SI by Method and Tier
    df_pivot_asi = df_success.pivot_table(
        values="Adjusted_SI", index="Method", columns="Tier", aggfunc="mean"
    )
    for tier in df_pivot_asi.columns:
        fig_perf.add_trace(
            go.Bar(
                name=tier,
                x=df_pivot_asi.index,
                y=df_pivot_asi[tier],
                showlegend=False,
            ),
            row=2,
            col=1,
        )

    # Runtime by Method
    for method in df_success["Method"].unique():
        method_data = df_success[df_success["Method"] == method]
        fig_perf.add_trace(
            go.Box(name=method, y=method_data["Runtime_Sec"], showlegend=False),
            row=2,
            col=2,
        )

    # Update layout
    fig_perf.update_yaxes(title_text="Recall", range=[0, 1.1], row=1, col=1)
    fig_perf.update_yaxes(title_text="Precision", range=[0, 1.1], row=1, col=2)
    fig_perf.update_yaxes(title_text="Adjusted SI", row=2, col=1)
    fig_perf.update_yaxes(title_text="Runtime (seconds)", row=2, col=2)
    fig_perf.update_xaxes(title_text="Method", row=1, col=1)
    fig_perf.update_xaxes(title_text="Method", row=1, col=2)
    fig_perf.update_xaxes(title_text="Method", row=2, col=1)
    fig_perf.update_xaxes(title_text="Method", row=2, col=2)

    fig_perf.update_layout(
        height=800,
        showlegend=True,
        barmode="group",
        legend=dict(title="Tier", x=1.05, y=1),
    )
    
    fig_perf.show()
else:
    print("No data available")

## Feature Selection Quality

Analyze the trade-off between correct features (n_correct) and false positives (n_extra).

In [5]:
if df is not None and len(df_success) > 0:
    # Create subplots
    fig_quality = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Feature Selection Quality: Correct vs False Positives",
            "Number of Solutions Found by Method",
        ),
    )

    # Scatter: Correct vs Extra features
    for method in df_success["Method"].unique():
        method_data = df_success[df_success["Method"] == method]
        fig_quality.add_trace(
            go.Scatter(
                x=method_data["n_correct"],
                y=method_data["n_extra"],
                mode="markers",
                name=method,
                opacity=0.6,
                marker=dict(size=10),
            ),
            row=1,
            col=1,
        )

    # Solutions found by method
    solutions_summary = df_success.groupby("Method")["Solutions_Found"].agg(
        ["mean", "std", "min", "max"]
    )

    for stat in ["mean", "std", "min", "max"]:
        fig_quality.add_trace(
            go.Bar(
                name=stat.capitalize(),
                x=solutions_summary.index,
                y=solutions_summary[stat],
                showlegend=(stat == "mean"),
            ),
            row=1,
            col=2,
        )

    # Update layout
    fig_quality.update_xaxes(
        title_text="Correct Features (n_correct)", row=1, col=1
    )
    fig_quality.update_yaxes(title_text="False Positives (n_extra)", row=1, col=1)
    fig_quality.update_xaxes(title_text="Method", row=1, col=2)
    fig_quality.update_yaxes(title_text="Number of Solutions", row=1, col=2)

    fig_quality.update_layout(height=500, showlegend=True, barmode="group")
    
    fig_quality.show()
else:
    print("No data available")

## Scalability Analysis

Examine how performance and runtime scale with problem size (N and P).

In [6]:
if df is not None and len(df_success) > 0:
    # Create subplots
    fig_scale = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=(
            "Scalability: Runtime vs Problem Size (P)",
            "Performance vs Problem Size (P)",
        ),
    )

    # Runtime vs Number of Features (P)
    for method in df_success["Method"].unique():
        method_data = df_success[df_success["Method"] == method]
        fig_scale.add_trace(
            go.Scatter(
                x=method_data["N_FEATURES"],
                y=method_data["Runtime_Sec"],
                mode="markers",
                name=method,
                opacity=0.6,
                marker=dict(size=10),
            ),
            row=1,
            col=1,
        )

    # Adjusted SI vs Number of Features
    for method in df_success["Method"].unique():
        method_data = df_success[df_success["Method"] == method]
        fig_scale.add_trace(
            go.Scatter(
                x=method_data["N_FEATURES"],
                y=method_data["Adjusted_SI"],
                mode="markers",
                name=method,
                opacity=0.6,
                marker=dict(size=10),
                showlegend=False,
            ),
            row=1,
            col=2,
        )

    # Update layout
    fig_scale.update_xaxes(title_text="Number of Features (P)", row=1, col=1)
    fig_scale.update_yaxes(title_text="Runtime (seconds)", type="log", row=1, col=1)
    fig_scale.update_xaxes(title_text="Number of Features (P)", row=1, col=2)
    fig_scale.update_yaxes(title_text="Adjusted Success Index", row=1, col=2)

    fig_scale.update_layout(height=500, showlegend=True)
    
    fig_scale.show()
else:
    print("No data available")

## Summary Statistics

Overall ranking and summary of method performance.

In [7]:
if df is not None and len(df_success) > 0:
    # Aggregate performance by method
    summary = (
        df_success.groupby("Method")
        .agg(
            {
                "Recall": ["mean", "std"],
                "Precision": ["mean", "std"],
                "Adjusted_SI": ["mean", "std"],
                "Runtime_Sec": ["mean", "median", "max"],
                "Solutions_Found": "mean",
            }
        )
        .round(3)
    )

    print("### Benchmark Summary by Method")
    display(summary)

    # Best method analysis
    best_recall = df_success.loc[df_success["Recall"].idxmax()]
    best_precision = df_success.loc[df_success["Precision"].idxmax()]
    best_asi = df_success.loc[df_success["Adjusted_SI"].idxmax()]
    fastest = df_success.loc[df_success["Runtime_Sec"].idxmin()]

    print("\n### Top Performers")
    print(f"\n🏆 **Best Recall**: {best_recall['Method']} ({best_recall['Recall']:.3f})")
    print(f"   - Tier: {best_recall['Tier']}, N={best_recall['N_SAMPLES']}, P={best_recall['N_FEATURES']}")
    
    print(f"\n🏆 **Best Precision**: {best_precision['Method']} ({best_precision['Precision']:.3f})")
    print(f"   - Tier: {best_precision['Tier']}, N={best_precision['N_SAMPLES']}, P={best_precision['N_FEATURES']}")
    
    print(f"\n🏆 **Best Adjusted SI**: {best_asi['Method']} ({best_asi['Adjusted_SI']:.3f})")
    print(f"   - Tier: {best_asi['Tier']}, N={best_asi['N_SAMPLES']}, P={best_asi['N_FEATURES']}")
    
    print(f"\n⚡ **Fastest**: {fastest['Method']} ({fastest['Runtime_Sec']:.3f}s)")
    print(f"   - Tier: {fastest['Tier']}, N={fastest['N_SAMPLES']}, P={fastest['N_FEATURES']}")
else:
    print("⚠️ No data available for analysis. Run benchmark first.")

### Benchmark Summary by Method


Recall     Precision     Adjusted_SI     Runtime_Sec                    \
         mean std      mean std        mean std        mean   median      max   
Method                                                                          
ALFESE  0.929 NaN     0.117 NaN       1.554 NaN     496.742  496.742  496.742   
GEMSS   1.000 NaN     1.000 NaN      14.286 NaN      14.932   14.932   14.932   

       Solutions_Found  
                  mean  
Method                  
ALFESE             6.0  
GEMSS              6.0


### Top Performers

🏆 **Best Recall**: GEMSS (1.000)
   - Tier: tier1, N=50, P=200

🏆 **Best Precision**: GEMSS (1.000)
   - Tier: tier1, N=50, P=200

🏆 **Best Adjusted SI**: GEMSS (14.286)
   - Tier: tier1, N=50, P=200

⚡ **Fastest**: GEMSS (14.932s)
   - Tier: tier1, N=50, P=200


---

## Notes

### Metrics Explained:
- **Recall**: Fraction of true features successfully recovered (higher is better)
- **Precision**: Fraction of selected features that are true positives (higher is better)
- **Success Index (SI)**: GEMSS-specific metric balancing coverage and accuracy
- **Adjusted SI**: SI weighted by precision (combines recall and precision)
- **Solutions Found**: Number of distinct feature sets discovered

### Usage:
1. Run the benchmark: `uv run python scripts/run_tier_sweep.py`
2. Results are saved to: `results/benchmark_results_YYYYMMDD_HHMMSS.csv`
3. Run all cells in this notebook to generate visualizations

### Tips:
- Use `df_success` to filter only successful runs
- Use `df[df['Tier'] == 'tier1']` to analyze specific tiers
- All plots are interactive - hover for details, zoom, pan, and download